In [ ]:
import os
import time
import numpy as np
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS


llm = ChatOpenAI(model='gpt-4o-mini')
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
from google.colab import userdata
import os

# Colab Secrets에서 'OPENAI_API_KEY'를 가져옵니다.
try:
    api_key = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = api_key
    print(f'Successfully loaded API key: {api_key[:10]}...')
except Exception as e:
    print('Colab Secrets에서 OPENAI_API_KEY를 찾을 수 없습니다. 왼쪽 열쇠 아이콘에서 설정을 확인해주세요.')

Successfully loaded API key: sk-proj-__...


In [ ]:
pip install langchain_community

In [ ]:
pip install langchain_text_splitters

In [ ]:
pip install langchain_openai

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
# langchain_text_splitters에서 직접 임포트

In [ ]:
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.

제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

주요 트렌드
RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

주요 기능
음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
자동 스케줄: 시간대별 기기 자동 제어
에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
보안 모드: 외출 시 자동 보안 설정
"""
}
SAMPLE_DIR = Path('sample_data')
SAMPLE_DIR.mkdir(exist_ok=True)

for filename, content in sample_texts.items():
  (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')
  # sample_data/product_manual.txt 등의 파일이 생성됨

## FAISS 벡터스토어와 Retriever 구성

In [ ]:
from pathlib import Path

data_dir = Path("sample_data")

files = {
    data_dir / "company_policy.txt" : "사내규정",
    data_dir / "product_manual.txt" : "제품매뉴얼",
    data_dir / 'ai_report.txt' : 'AI보고서'
}

documents = []
for fpath, category in files.items():
  text = fpath.read_text()
  for section in text.strip().split('\n\n'):
    if section.strip():
      documents.append(Document(
          page_content = section.strip(),
          metadata = {'category': category, 'source': fpath.name}
      ))

print(f"총 {len(documents)}개의 문서 세그먼트가 생성되었습니다.")

총 13개의 문서 세그먼트가 생성되었습니다.


In [ ]:
documents[0]

Document(metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='주식회사 모두의연구소 사내 규정')

In [ ]:
pip install faiss-cpu

In [ ]:
vectorstore = FAISS.from_documents(documents, embeddings) # 문서를 가져와서 벡터화함

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

In [ ]:
vectorstore.save_local('faiss_docs') # 벡터를 저장함

In [ ]:
loaded_vs = FAISS.load_local('faiss_docs', embeddings, allow_dangerous_deserialization=True)
# pkl 파일-> 2020년대 초반에는 쓰였는데 언제부터 안쓰임..

In [ ]:
loaded_vs.index.ntotal # document 개수(임베딩 후 들어있는 벡터화된 문서 개수)

13

- retriever = vectorstore.as_retriever ⭐
- retriever.invoke():질문(query)과 가장 관련성이 높은 문서 조각들을 벡터 저장소에서 찾아오는 역할

In [ ]:
retriever.invoke('재택근무 규정')  # 질문(query)과 가장 관련성이 높은 문서 조각들을 벡터 저장소에서 찾아오는 역할

[Document(id='bb87d1d2-e75b-499e-9669-53f8ee7b28c6', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.'),
 Document(id='1f6d743e-3ee9-4dd3-b2b6-294ed574261d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.'),
 Document(id='63f02de9-4be5-435b-b749-6fd94f1bad76', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.')]

## Retriever 기반 RAG 체인 구성

In [ ]:
# LCEL: LangCahin Expression Language
# | | 등의 기호로 이어줌
# prompt | llm | StrOutputParser()

In [ ]:
simple_prompt = ChatPromptTemplate.from_messages([
    ('user','{question}')  # ⭐ 튜플임!
])

simple_chain = simple_prompt | llm | StrOutputParser()

In [ ]:
result = simple_chain.invoke({'question': '대한민국의 수도는?'})

In [ ]:
result

'대한민국의 수도는 서울입니다.'

In [ ]:
class DocStore:

  def retrieve

  def generate(query)
    retrieved = retrieve(query) # 쿼리를 통해 연관된 문서를 뽑아옴

    # 했던것
    # SystemMessage : 아래 문서를 참고해서 주세요~
    # HumanMessage : "".join(retrieve) 등으로 엮어서 답변을 받음

In [ ]:
# RAG

# 쿼리 입력 -> retriever가 검색을 해오고 -> llm에 넣어야 하는데

# llm : ("쿼리,, 재택근무 규정이 어떻게 되나요?", 'retrieved documents')

# rag_chain = simple_prompt: 재택근무 규정이 어떻게 되나요 | retriever 결과 | llm (llm에는 simple_prompt + retriever 같이 들어가야됨) | Parser()
  # 여기서 simple_prompt만 들어가면 안됨

# 그냥 쓰면, llm에 simple_prompt가 들어가던가, retriever 결과만 들어가게됨
# 그럼 어떻게 하냐?

# ⭐ RunnablePassthrough

from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()



'안녕하세요'

In [ ]:
passthrough.invoke("안녕하세요") # 랭체인은 invoke() 함수로 입력값을 받음 (함수 내부 인자로 전달)
                                # retriever도 동일하게 invoke()를 쓰던것처럼

'안녕하세요'

In [ ]:
passthrough.invoke({'a':1})

{'a': 1}

In [ ]:
def format_docs(docs): # 리트리버가 가져온 문자열을 포매팅 해주는 함수
                   # 리트리버는 리스트 형태로 가져옴, llm chain에 입력으로 들어가기 힘듦,

    return '\n-\n'.join(
        f"[{d.metadata.get('category', '')}] {d.page_content}" # doc의 메타데이터와 page_content를 for 문을 이용해 포매팅해준다 는 뜻
        for d in docs
    )


In [ ]:
docs = retriever.invoke('재택근무')

In [ ]:
formatted = format_docs(docs)

In [ ]:
print(formatted) # 하이픈 들어있고, [카테고리] 들어있고, content 들어가있음(제 2조... 부터 content 부분임)

[사내규정] 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
-
[사내규정] 제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
-
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


리트리버 체인

In [ ]:
retriever_chain = retriever | format_docs # 이게 함수를 그냥 파이프(chain)에 붙이니까 바로 되네..
                                          # ⭐ 커스텀 함수인데, 체인에 붙여 사용 가능
context_text = retriever_chain.invoke("재택근무")

In [ ]:
print(context_text)

[사내규정] 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
-
[사내규정] 제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
-
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


In [ ]:
# 아래 랭체인 코드와 동일한 기능임

question = '"재택근무 규정이 어떻게 되나요?'
docs = vetorstore.similarity_search(question, k=3)
context = '\n-\n'.join(
    f"[{d.metadata.get('category', '')}] {d.page_content}"
    for d in docs)

messages = [
    System...
    Human
]

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요."),
    ("user", "{question}")

])

rag_chain = ({"context": retriever | format_docs, # 여기서 question에 먼저 받고 RunnablePassthrough로 패싱, retriever와 format_docs에 먼저 들어감
              "question": RunnablePassthrough()} # 얘는 rag_prompt가 원하는 형식에 맞게, context와 question으로 구성됨
            | rag_prompt
            | llm
            | StrOutputParser()
            )

In [ ]:
answer = rag_chain.invoke("재택근무 규정이 어떻게 되나요?")
print(answer)

재택근무는 주 2회까지 가능하다는 규정이 있습니다.


In [ ]:
question_list = ['스마트홈 허브 초기 설정 방법', 'ai 산업 성장률', '교육비 지원 한도']
for q in question_list:
  print(f"Q: {q}")
  print(f"A: {rag_chain.invoke(q)}")
  print("="*6)


Q: 스마트홈 허브 초기 설정 방법
A: 스마트 홈 허브 v3.0의 초기 설정 방법은 다음과 같습니다:

1. **전원 연결**: 스마트 홈 허브를 전원에 연결합니다.

2. **앱 다운로드**: 스마트폰 또는 태블릿에 스마트 홈 허브 전용 앱을 다운로드합니다. (앱 스토어에서 "스마트 홈 허브" 검색)

3. **계정 생성 또는 로그인**: 앱을 실행한 후, 계정을 생성하거나 기존 계정으로 로그인합니다.

4. **허브 연결**: 앱에서 허브를 찾아 연결을 시도합니다. 이 과정에서 Wi-Fi 네트워크에 연결해야 할 수 있습니다.

5. **기기 추가**: 연결된 허브를 통해 제어할 스마트 기기를 추가합니다. 음성 명령이나 앱을 통해 기기를 추가할 수 있습니다.

6. **기기 설정**: 추가된 기기를 설정하여 원하는 대로 작동하도록 합니다. 필요한 경우 자동 스케줄을 설정합니다.

7. **말하기 명령 확인**: 음성 인식 기능을 테스트하여 "허브야, 거실 조명 켜줘"와 같은 명령을 사용해봅니다.

8. **에너지 모니터링 및 보안 설정**: 에너지 사용량을 모니터링하고, 외출 시 보안 모드를 설정합니다.

이 절차를 따르면 스마트 홈 허브의 초기 설정이 완료됩니다. 추가적인 문의사항은 사용자 매뉴얼을 참고하시기 바랍니다.
Q: ai 산업 성장률
A: 2024년 인공지능 산업의 성장률은 전년 대비 35% 증가했습니다. 2024년에는 약 5,000억 달러 규모에 도달했으며, 특히 생성형 AI 분야가 전체 성장의 60%를 견인했습니다.
Q: 교육비 지원 한도
A: 사내규정에 따르면, 외부 컨퍼런스 참석비는 연 200만원까지 지원됩니다. 또한, 온라인 학습 플랫폼 이용료는 전액 지원됩니다.


### 실습, rag_chain을 만들어 보세요, k=1로 새 체인을 만들어 비교를 해보세요.

In [ ]:
def format_docs(docs): # 리트리버가 가져온 문자열을 포매팅 해주는 함수
                   # 리트리버는 리스트 형태로 가져옴, llm chain에 입력으로 들어가기 힘듦,

    return '\n-\n'.join(
        f"[{d.metadata.get('category', '')}] {d.page_content}" # doc의 메타데이터와 page_content를 for 문을 이용해 포매팅해준다 는 뜻
        for d in docs
    )

# ====================================

retriever = vectorstore.as_retriever(search_kwargs={'k':1}) # 리트리버, k=1 설정

# rag_prompt 정의 (템플릿)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요."),
    ("user", "{question}")

])

rag_chain = ({"context": retriever | format_docs, # 여기서 question에 먼저 받고 RunnablePassthrough로 패싱, retriever와 format_docs에 먼저 들어감
              "question": RunnablePassthrough()} # 얘는 rag_prompt가 원하는 형식에 맞게, context와 question으로 구성됨
            | rag_prompt
            | llm
            | StrOutputParser()
            )


In [ ]:
answer = rag_chain.invoke("재택근무 규정이 어떻게 되나요?")
print(answer)

재택근무는 주 2회까지 가능합니다.


In [ ]:
question_list = ['스마트홈 허브 초기 설정 방법', 'ai 산업 성장률', '교육비 지원 한도']
for q in question_list:
  print(f"Q: {q}")
  print(f"A: {rag_chain.invoke(q)}")
  print("="*6)


Q: 스마트홈 허브 초기 설정 방법
A: 스마트홈 허브 v3.0의 초기 설정 방법은 다음과 같습니다:

1. **전원 연결**: 스마트홈 허브의 전원 어댑터를 전원 콘센트에 연결한 후, 허브의 전원 버튼을 눌러 켭니다.

2. **모바일 앱 다운로드**: 스마트폰에 '스마트홈 앱'을 다운로드합니다. 앱 스토어(안드로이드용 구글 플레이스토어 또는 iOS의 앱 스토어)에서 검색하여 설치합니다.

3. **계정 생성 또는 로그인**: 앱을 실행하고, 계정을 새로 생성하거나 기존 계정으로 로그인합니다.

4. **허브 연결**: 앱 내에서 '허브 추가' 또는 '장치 추가' 메뉴를 선택합니다. 이후 화면의 지시에 따라 스마트홈 허브와 앱을 연결합니다. 이 과정에서 허브가 와이파이에 연결될 수 있도록 설정해줍니다.

5. **스마트 기기 추가**: 허브가 정상적으로 연결되면, 스마트홈 앱에서 다양한 스마트 기기를 추가하여 설정할 수 있습니다. '장치 추가' 옵션을 선택하고, 추가할 기기의 종류를 선택한 후 지시에 따라 진행합니다.

6. **초기 설정 완료**: 모든 설정이 완료되면, 각 스마트 기기를 제어하고 자동화를 설정할 수 있습니다. 

이 외에도 필요에 따라 다양한 설정을 조정할 수 있으니, 사용자 매뉴얼을 참고하여 추가적인 기능을 활용해 보세요. 

문서 내용에 따라 다를 수 있으므로, 보다 정확한 정보는 사용자 매뉴얼을 확인하시기 바랍니다.
Q: ai 산업 성장률
A: 2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했습니다. 이 중 생성형 AI 분야가 전체 성장의 60%를 견인했습니다.
Q: 교육비 지원 한도
A: 교육비 지원 한도는 외부 컨퍼런스 참석비로 연 200만원까지 지원됩니다. 또한, 온라인 학습 플랫폼 이용료는 전액 지원됩니다.


In [ ]:
# 어떤 문서에서 가져왔는지...표시하려면
# RunnableParallel을 쓰면 됨
# RunnableParallel은 여러 작업을 동시에(병렬로) 실행하고, 그 결과들을 하나의 딕셔너리 형태로 묶어주는 역할

# rag_prompt 정의 (템플릿)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요(문서에 없으면 없다고 말해주세요)."),
    ("user", "{question}")

])

rag_chain = ({"context": retriever | format_docs, # 여기서 question에 먼저 받고 RunnablePassthrough로 패싱, retriever와 format_docs에 먼저 들어감
              "question": RunnablePassthrough()} # 얘는 rag_prompt가 원하는 형식에 맞게, context와 question으로 구성됨
            | rag_prompt
            | llm
            | StrOutputParser()
            )


## 고급 체인 기능 — RunnableParallel과 유사도 평가

In [ ]:
from langchain_core.runnables import RunnableParallel

In [ ]:
rag_chain_with_soruces = RunnableParallel(
    answer = rag_chain, # rag_chain의 결과가 answer에 들어갈거고
    source_document = retriever
)

# 입력 공유: invoke('질문')를 호출하면, 그 '질문' 문자열이 answer와 source_document 양쪽으로 동시에 전달
  # answer: 이전에 정의한 rag_chain이 실행되어 최종 답변 문자열을 생성
  # source_document: retriever가 실행되어 질문과 관련된 원본 Document 객체 리스트를 가져옴

In [ ]:
rag_chain_with_soruces.invoke('재택근무 규정이 어떻게 되나요?') # 딕셔너리 형태로 결과가 나옴

{'answer': '재택근무는 주 2회까지 가능하다는 규정이 있습니다.',
 'source_document': [Document(id='f30bb5be-3a43-416e-9c94-acf3e38a12c5', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.')]}

In [ ]:
result = rag_chain_with_soruces.invoke('재택근무 규정이 어떻게 되나요?')

for doc in result['source_document']:
  print(f"[{doc.metadata['category']}] {doc.page_content[:100]}")

[사내규정] 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.


### 실습, 답변을 streaming으로 한번 받아보세요.

In [ ]:
# ⭐stream은 lang_chain 객체의 메서드임. (Prompt, llm 및 Parser 객체를 LCEL로 표현한 것이 chain 객체임)
  # Prompt + LLM + Parser 등 여러 Runnable 객체들을 | (파이프) 기호로 엮어서 만든 최종 결과물이 바로 Chain 객체
question = '재택근무 규정이 어떻게 되나요?'

for chunk in rag_chain.stream(question):
    print(chunk, end='', flush=True)

재택근무는 주 2회까지 가능하다는 규정이 있습니다. 더 자세한 사항이 필요하시면 추가로 질문해 주세요!

In [ ]:
rag_chain.invoke("2022년 월드컵 우승팀은 어디인가요?")

'죄송하지만, 해당 정보는 제공된 문서에 포함되어 있지 않습니다.'

In [ ]:
# RunnableBranch # if문처럼 결과가 A일때는, B일때는 등을 설정

In [ ]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
# 함수단에서 막을 수 있는지


In [ ]:
def check_and_prepare(question): # 질문이 들어가면
  results = vectorstore.similarity_search_with_score(question, k=3)
  docs = [doc for doc. _ in results]
  return {
      'question' : question,
      'context' : format_docs(docs), # 아까 정의한 함수
      'score' : results[0][1]
  }
threshold =1.3
safe_chain = (
    # 함수를 실행하는 Runnable은 RunnableLambda임
    RunnableLambda(check_and_prepare) # 체인을 쓸때, 입력값에 대한건 파이프에 안들어감 (이전 체인의 출력값이 인풋으로 들어감),모든 인풋은 chain.invoke()의 ()에 처음 들어감
    | RunnableBranch(
        (lambda x: x['score'] > threshold, lambda x: '해당 정보가 없습니다.'),
        rag_prompt | llm     | StrOutputParser()
  )
)

safe_chain.invoke('재택근무 규정을 알려주세요')

'재택근무는 주 2회까지 가능하다는 규정이 있습니다.'

### 실습: RunnableParallel: answer, source_documents, 검색된 텍스트 context를 동시에 리턴하는 체인

In [ ]:
# rag_prompt 정의 (템플릿)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요(문서에 없으면 없다고 말해주세요)."),
    ("user", "{question}")

])

rag_chain = ({"context": retriever | format_docs,
              "question": RunnablePassthrough()}
            | rag_prompt
            | llm
            | StrOutputParser()
            )

rag_chain_with_soruces = RunnableParallel(
    answer = rag_chain,
    source_document = retriever,
    context = retriever | format_docs
)

In [ ]:
rag_chain_with_soruces.invoke("뭐해")

{'answer': '저는 스마트 홈 허브 v3.0 사용자 매뉴얼을 기반으로 질문에 답변하는 도우미입니다. 궁금한 점이 있으시면 물어보세요!',
 'source_document': [Document(id='99b56fbe-66dd-4da8-a119-8fbd5bc25e8a', metadata={'category': '제품매뉴얼', 'source': 'product_manual.txt'}, page_content='스마트 홈 허브 v3.0 사용자 매뉴얼')],
 'context': '[제품매뉴얼] 스마트 홈 허브 v3.0 사용자 매뉴얼'}

In [ ]:
# vectorstore.similarity_search_with_score(question, k=3)로 스코어값을 이용함
  # threshold는 이걸 이용한거임
  # 해당 함수의 score는 벡터간의 거리기반임, 값이 높으면 '멀다'는 뜻 (코사인 유사도와 좀 반대개념)
# 이걸 안사용하고 retriever 함수만 이용한다면 그냥 page_content를 임베딩하고, query와의 코사인 유사도 + threshold 이용해서
# 관계없는 답변은 쳐내면 됨

retriever.invoke('2002년 월드컵')

[Document(id='a339b866-b1ce-4a06-b3e1-d045537293cf', metadata={'category': 'AI보고서', 'source': 'ai_report.txt'}, page_content='2024년 인공지능 산업 동향 보고서')]

In [ ]:
for doc, score in vectorstore.similarity_search_with_score("재택근무 규정", k=3):
  print(f"{doc.metadata['category']} {doc.page_content}")
  print(score)

사내규정 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
0.9916115
사내규정 제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.
1.2501051
사내규정 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.
1.2635785


In [ ]:
test_queries = [
    {'query': "재택근무 몇 회?"},
    {'query': "스마트홈 초기 설정"},
    {'query': "RAG 기술 동향"}
]

for tq in test_queries:
  results = vectorstore.similarity_search_with_score(tq['query'], k=1)
  #print(f'results:{results}')
  top_cat = results[0][0].metadata['category'] # doc과 score 즉, 0번째 제일 스코어 좋은것?
  context = results[0][0].page_content
  top_score = results[0][1]
  print(f"{tq}: top category : {top_cat}, {top_score}, {context}")

  # 좋은 답변을 생성하려면 답변을 하나씩 뽑아보면서 threshold 등의 하이퍼 파라미터를 설정하는 것이 좋음
  # ⭐ 검색 품질 관점!

{'query': '재택근무 몇 회?'}: top category : 사내규정, 0.9775654673576355, 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
{'query': '스마트홈 초기 설정'}: top category : 제품매뉴얼, 0.8590512275695801, 스마트 홈 허브 v3.0 사용자 매뉴얼
{'query': 'RAG 기술 동향'}: top category : AI보고서, 1.1314737796783447, 2024년 인공지능 산업 동향 보고서


In [ ]:
# 실습! vectorstore.similarity_search_with_score
# query를 입력하고 -> threshold 설정 -> 답변할 수 없다. / 해당 문서 내용

def verify_query(query, threshold):
    # 1. 유사도 검색 수행
    results = vectorstore.similarity_search_with_score(query, k=1)

    if not results:
        return "검색 결과가 없습니다."

    doc, score = results[0]

    # 2. threshold 비교 (L2 거리는 낮을수록 유사함)
    if score > threshold:
        return f"[점수: {score:.4f}] 해당 정보가 문서에 없습니다."
    else:
        return f"[점수: {score:.4f}] 관련 내용: {doc.page_content}"

# 테스트
print(verify_query("재택근무 규정", 1.2))
print(verify_query("2002년 월드컵 우승팀", 1.2))

[점수: 0.9920] 관련 내용: 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
[점수: 1.6081] 해당 정보가 문서에 없습니다.


In [ ]:
results = vectorstore.similarity_search_with_score(tq['query'], k=1)

results # doc과 score가 나옴

[(Document(id='a339b866-b1ce-4a06-b3e1-d045537293cf', metadata={'category': 'AI보고서', 'source': 'ai_report.txt'}, page_content='2024년 인공지능 산업 동향 보고서'),
  np.float32(1.1330541))]

# 📝 지금까지의 학습 내용 요약

본 실습에서는 **LangChain**과 **FAISS**를 활용하여 외부 지식을 참고해 답변하는 **RAG(Retrieval-Augmented Generation) 시스템**의 전 과정을 구현했습니다.

### 1. 데이터 로딩 및 전처리
- `pathlib`을 이용해 로컬 텍스트 파일을 읽고, `langchain_core.documents.Document` 형식으로 변환하였습니다.
- 각 문서에 `category`, `source` 등의 메타데이터를 부여하여 출처 추적이 가능하게 설정했습니다.

### 2. 임베딩 및 벡터 스토어 (Vector Store)
- `OpenAIEmbeddings`(`text-embedding-3-small`)를 사용하여 텍스트의 의미를 수치화했습니다.
- `FAISS`를 통해 고성능 벡터 검색 엔진을 구축하고, `save_local` 및 `load_local`로 인덱스를 영구 저장하는 방법을 확인했습니다.

### 3. LCEL을 이용한 RAG 체인 구축
- **Retriever**: `vectorstore.as_retriever(search_kwargs={'k': 3})`로 검색기 설정.
- **Prompt**: `ChatPromptTemplate`을 사용해 시스템 메시지와 유저 질문 템플릿 정의.
- **Chain**: `RunnablePassthrough`로 입력을 전달하고, `|` (Pipe) 연산자로 컴포넌트를 연결하는 **LCEL(LangChain Expression Language)** 구조를 완성했습니다.

### 4. 고급 기능 및 최적화
- **출처 반환**: `RunnableParallel`을 사용하여 답변(`answer`)과 참고 문서(`source_document`)를 동시에 출력.
- **스트리밍**: `chain.stream()`을 통해 답변이 생성되는 대로 실시간 출력 구현.
- **신뢰도 제어**: `similarity_search_with_score`를 통해 벡터 간 거리를 측정하고, 일정 `threshold`(임계값)를 넘으면 답변을 거부하는 로직을 추가하여 할루시네이션(Hallucination)을 방지했습니다.